# Navier–Lamé reduction, the operator way: $\nabla\cdot T = \mu\,\nabla\cdot\nabla u + \nabla((\lambda+\mu)\nabla\cdot u)$

The isotropic Hooke stress for a displacement field $u$ is
$T = \lambda(\nabla\cdot u)\,I + \mu(\nabla u + (\nabla u)^{\mathsf{T}})$, and the
equilibrium divergence $\nabla\cdot T$ is the elastic operator acting on $u$.  We
reduce it **as performed** — expand $\nabla$ only, keep $u$ abstract, apply the
Leibniz $\partial$'s, fold $e_i\cdot I$, reassemble the operators, reverse the
distribution — then verify it componentwise (vibe 000080, Increment 8).

In [ ]:
import tender as t
import tender.derivation as td

ws = t.Workspace()
u = ws.field("u", 1)
nabla = t.nabla(ctx=ws.ctx)
I = t.identity(ws.ctx)
lam = t.tensor(r"\lambda", 0, ctx=ws.ctx)
mu = t.tensor(r"\mu", 0, ctx=ws.ctx)

## 1. Coordinate-free, $u$ abstract

`@` is the dot, `*` the tensor product.  $\nabla\cdot T$ is built chart-free — no
$u_x$ components appear, only $\nabla u$ and $\nabla\cdot u$.

In [ ]:
T = lam * (nabla @ u) * I + mu * (nabla * u + (nabla * u).transpose())
divT = nabla @ T
divT

## 2. Expand $\nabla$ only — the free-index interior

`expand_nabla` lowers each $\nabla$ to the frame form $e_i\,\partial_i$ and applies
the Leibniz $\partial$'s; the inner $\nabla\cdot u$ hidden inside $(\nabla\cdot u)I$
is resolved before the outer $\partial$.  `contract_identity` folds every
$e_i\cdot I\to e_i$ — still abstract in $u$.

In [ ]:
x, y, z = ws.coords("x", "y", "z")
cart = ws.chart(ws.wcs(), [x, y, z], [x, y, z])
interior = td.contract_identity(td.canonicalize(cart.expand_nabla(divT)))
interior

## 3. Reassemble the $\partial$'s back into $\nabla$ operators

`reassemble_nabla` reads each frame-indexed $\partial$'s role and folds it back
into $\nabla$ operators, carrying the Lamé coefficients $\lambda,\mu$ through, for
$\nabla\cdot T = \lambda\nabla(\nabla\cdot u) + \mu\nabla(\nabla\cdot u) + \mu\nabla\cdot\nabla u$.

In [ ]:
reass = cart.reassemble_nabla(td.canonicalize(interior))
assert r"\lambda" in reass.latex() and r"\mu" in reass.latex()  # Lamé coeffs survived
td.collect_terms(reass)

## 4. Reverse the distribution — the Navier–Lamé endpoint

`factor_common` pulls the shared $\nabla\cdot u$ out of the $\lambda$- and
$\mu$-gradients and then hoists the constant $(\lambda+\mu)$ fully outside the
gradient ($\nabla(cX)=c\nabla X$, valid since $\nabla c=0$), giving the clean form
$\nabla\cdot T = \mu\,\nabla\cdot\nabla u + (\lambda+\mu)\,\nabla(\nabla\cdot u)$
($=\mu\,\Delta u + (\lambda+\mu)\nabla(\nabla\cdot u)$).  The hoist is an
operator-linearity rewrite, so the endpoint is verified componentwise in \S5.

In [ ]:
nl = td.factor_common(td.collect_terms(reass))
nl  # correctness is proven componentwise in section 5 (below)

## 4b. The textbook form $T = \lambda\,\mathrm{tr}(\varepsilon)I + 2\mu\varepsilon$

With $\varepsilon = \mathrm{sym}(\nabla u) = (\nabla u + (\nabla u)^{\mathsf{T}})/2$, the
scalar $/2$ rides out through the whole reduction — the constant-denominator
differentiation rule keeps the $\partial$-mark indices linked — so the textbook
stress reduces to the *same* clean Navier–Lamé endpoint.

In [ ]:
T_std = lam * (nabla @ u) * I + t.scalar(2, ctx=ws.ctx) * mu * td.sym(nabla * u)
reass_std = cart.reassemble_nabla(td.canonicalize(
    td.contract_identity(td.canonicalize(cart.expand_nabla(nabla @ T_std)))))
nl_std = td.factor_common(td.collect_terms(reass_std))
assert nl_std.latex() == nl.latex()   # same endpoint as the explicit form
nl_std

## 5. Verify the endpoint, component by component

Both sides are coordinate-free vectors, so the identity holds in every frame —
tender confirms $\nabla\cdot T = \mu\,\nabla\cdot\nabla u + (\lambda+\mu)\nabla(\nabla\cdot u)$
in a Cartesian and a cylindrical frame.

In [ ]:
def is_zero(chart, e):
    return td.simplify_scalars(td.canonicalize(chart.expand(e))).latex() == "0"

def verify(chart, u, lam, mu, I):
    gradu = chart.grad(u)
    T = lam * chart.div(u) * I + mu * (gradu + gradu.transpose())
    lhs = chart.components(chart.div(T))                                   # ∇·T
    rhs = chart.components(mu * chart.div(chart.grad(u))
                           + (lam + mu) * chart.grad(chart.div(u)))        # Navier–Lamé
    return all(is_zero(chart, chart.expand(lhs[i]) - chart.expand(rhs[i]))
               for i in range(3))

ws2 = t.Workspace(); u2 = ws2.field("u", 1)
lam2 = t.tensor(r"\lambda", 0, ctx=ws2.ctx); mu2 = t.tensor(r"\mu", 0, ctx=ws2.ctx)
I2 = t.identity(ws2.ctx)
r, th, zc = ws2.coords("r", r"\theta", "z", nonneg=("r",))
cyl = ws2.chart(ws2.wcs(), [r, th, zc], [r*t.cos(th), r*t.sin(th), zc])
print("Cartesian  :", "all 3 components equal" if verify(cart, u, lam, mu, I) else "MISMATCH")
print("Cylindrical:", "all 3 components equal" if verify(cyl, u2, lam2, mu2, I2) else "MISMATCH")

## 6. The clean path — `chart.evaluate` of the coordinate-free $\nabla\cdot T$

No re-typing with `chart.grad/div/rot`: hand the invariant $\nabla\cdot T$ (with `u` abstract and the coordinate-free $\nabla$) straight to `chart.evaluate`, which lowers each $\nabla$-combination to the curvilinear-correct operators (vibe 000084). It matches the same endpoint in every frame.

In [ ]:
def verify_evaluate(chart, nabla, u, lam, mu, I):
    T = lam * (nabla @ u) * I + mu * (nabla * u + (nabla * u).transpose())
    lhs = chart.components(chart.evaluate(nabla @ T))                       # ∇·T in-chart
    rhs = chart.components(mu * chart.div(chart.grad(u))
                           + (lam + mu) * chart.grad(chart.div(u)))
    return all(is_zero(chart, chart.expand(lhs[i]) - chart.expand(rhs[i]))
               for i in range(3))

nabla2 = t.nabla(ctx=ws2.ctx)
print("Cartesian  :", "all 3 components equal" if verify_evaluate(cart, nabla, u, lam, mu, I) else "MISMATCH")
print("Cylindrical:", "all 3 components equal" if verify_evaluate(cyl, nabla2, u2, lam2, mu2, I2) else "MISMATCH")